In [1]:
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

In [2]:
X = np.load("X.npy")
y = np.load("y.npy")

print("X Shape:", X.shape)
print("y Shape:", y.shape)

print("Unique Labels:", np.unique(y))

X Shape: (8249, 512)
y Shape: (8249,)
Unique Labels: [0 1]


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Set:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nTesting Set:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

Training Set:
X_train: (6599, 512)
y_train: (6599,)

Testing Set:
X_test: (1650, 512)
y_test: (1650,)


In [5]:
# Convert features to float tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

# Convert labels to long tensors
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nData Types:")
print("X_train:", X_train.dtype)
print("y_train:", y_train.dtype)

X_train: torch.Size([6599, 512])
X_test : torch.Size([1650, 512])
y_train: torch.Size([6599])
y_test : torch.Size([1650])

Data Types:
X_train: torch.float32
y_train: torch.int64


In [6]:
# Create TensorDatasets
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

# Create DataLoaders
batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("Training Samples:", len(train_dataset))
print("Testing Samples :", len(test_dataset))

print("Training Batches:", len(train_loader))
print("Testing Batches :", len(test_loader))

Training Samples: 6599
Testing Samples : 1650
Training Batches: 207
Testing Batches : 52


In [7]:
class EEGCNN(nn.Module):
    def __init__(self):
        super(EEGCNN, self).__init__()

        self.conv = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(16, 32, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 64, 128),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(128, 2)
        )

    def forward(self, x):
        x = x.unsqueeze(1)      # (batch_size, 1, 512)
        x = self.conv(x)
        x = self.fc(x)
        return x

model = EEGCNN()

print(model)

EEGCNN(
  (conv): Sequential(
    (0): Conv1d(1, 16, kernel_size=(5,), stride=(1,), padding=(2,))
    (1): ReLU()
    (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv1d(16, 32, kernel_size=(5,), stride=(1,), padding=(2,))
    (4): ReLU()
    (5): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv1d(32, 64, kernel_size=(5,), stride=(1,), padding=(2,))
    (7): ReLU()
    (8): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4096, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=128, out_features=2, bias=True)
  )
)


In [8]:
# Select device
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Using Device:", device)

# Move model to device
model = model.to(device)

# Loss Function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

Using Device: mps


In [9]:
import torch

print(torch.__version__)
print(torch.backends.mps.is_available())
print(torch.backends.mps.is_built())

2.8.0
True
True


In [10]:
epochs = 20

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:

        # Move data to device
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # Forward pass
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}]  Loss: {running_loss/len(train_loader):.4f}")

Epoch [1/20]  Loss: 0.5868
Epoch [2/20]  Loss: 0.4827
Epoch [3/20]  Loss: 0.4277
Epoch [4/20]  Loss: 0.4114
Epoch [5/20]  Loss: 0.3753
Epoch [6/20]  Loss: 0.3522
Epoch [7/20]  Loss: 0.3268
Epoch [8/20]  Loss: 0.2962
Epoch [9/20]  Loss: 0.2616
Epoch [10/20]  Loss: 0.2399
Epoch [11/20]  Loss: 0.2152
Epoch [12/20]  Loss: 0.1940
Epoch [13/20]  Loss: 0.1671
Epoch [14/20]  Loss: 0.1471
Epoch [15/20]  Loss: 0.1304
Epoch [16/20]  Loss: 0.1152
Epoch [17/20]  Loss: 0.0984
Epoch [18/20]  Loss: 0.0889
Epoch [19/20]  Loss: 0.0756
Epoch [20/20]  Loss: 0.0804


In [11]:
model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        outputs = model(X_batch)

        _, predicted = torch.max(outputs, 1)

        y_true.extend(y_batch.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

# Metrics
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)

print(f"Test Accuracy : {accuracy*100:.2f}%")
print(f"Precision     : {precision*100:.2f}%")
print(f"Recall        : {recall*100:.2f}%")
print(f"F1 Score      : {f1*100:.2f}%")

print("\nConfusion Matrix:")
print(cm)

Test Accuracy : 80.79%
Precision     : 85.42%
Recall        : 85.81%
F1 Score      : 85.61%

Confusion Matrix:
[[390 161]
 [156 943]]
